# Matdoria Review count Pick 기준 분석 (v2)

현재 빌드의 로컬 score snapshot만 읽어 출처 커버리지, 임계값별 Pick 수, 상위 선정 근거를 확인합니다. 데이터 수집이나 API 호출은 하지 않습니다. 점수는 Matdoria 편집 규칙의 결과이며 객관적인 맛 평가가 아닙니다.

가중치 변경 실험은 `data/curation/matdoria_pick_score_v1.json`을 복사해 버전을 올린 뒤 `python scripts/score_matdoria_pick.py --profile <profile.json>`로 별도 snapshot을 생성해 비교하세요.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
history_path = ROOT / 'data/processed/build_history.jsonl'
history = [json.loads(line) for line in history_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if not history:
    raise FileNotFoundError('No build history. Run: python scripts/export_places.py')
build_info = history[-1]
snapshot_path = ROOT / build_info['scoreSnapshot']
snapshot = json.loads(snapshot_path.read_text(encoding='utf-8'))
places = snapshot['places']
print('snapshot:', snapshot_path.relative_to(ROOT))
print('profile:', snapshot['metadata']['profileId'], snapshot['metadata']['profileVersion'])
print('built:', snapshot['metadata']['buildTimestamp'])
print('candidate places:', snapshot['metadata']['candidatePlaceCount'])
print('selected places:', snapshot['metadata']['selectedPlaceCount'])

## 출처별 데이터 커버리지

`unknown`은 입력에 해당 신호가 없다는 뜻이고 감점하지 않습니다. `not_applicable`은 원본 기록은 있지만 프로필에서 점수화하지 않는 유형(예: 음식 골목 지정)입니다.

In [ ]:
for name, item in snapshot['coverage'].items():
    known = item['knownEvidencePlaces']
    scored = item['scoredPlaces']
    total = item['candidatePlaces']
    rate = 100 * item['scoredPlaceRate'] if item['scoredPlaceRate'] is not None else 0
    print(f'{name:24} scored={scored:4}/{total:<4} known={known:4} coverage={rate:5.1f}%')

## 임계값 민감도

K/G Review threshold 조합을 바꿨을 때 Pick 수와 count 커버리지가 어떻게 움직이는지 확인합니다. Review 수는 인기도 신호이며 맛 순위가 아닙니다.

In [ ]:
counts = [p['dimensions']['popularity']['counts'] for p in places]
both_known = sum(c['kakaoReviewCount'] is not None and c['googleUserRatingCount'] is not None for c in counts)
print(f'Both platform counts known: {both_known}/{len(places)}')
for k_threshold in [25, 50, 75, 100, 150]:
    for g_threshold in [100, 250, 300, 500]:
        eligible = sum(
            c['kakaoReviewCount'] is not None and c['googleUserRatingCount'] is not None
            and c['kakaoReviewCount'] >= k_threshold and c['googleUserRatingCount'] >= g_threshold
            for c in counts
        )
        print(f'K >= {k_threshold:>3}, G >= {g_threshold:>3}: {eligible:>3} places')

## 상위 선정 장소와 근거

Pick 항목별 K/G Review 수와 선정 기준 충족 여부를 확인합니다. 두 count snapshot이 없는 장소는 후보 자격을 판단할 수 없으며 낮은 품질로 간주하지 않습니다.

In [ ]:
for place in [p for p in places if p['selected']][:30]:
    popularity = place['dimensions']['popularity']
    counts = popularity['counts']
    print(f"{place['name']} | K Review {counts['kakaoReviewCount']} | G Review {counts['googleUserRatingCount']} | thresholds {popularity['thresholds']}")